In [6]:
import pandas as pd
import numpy as np
import editdistance
from skeleton import Skeletonizer, explode_nsi
from diverse_sample import select_diverse_captions

In [2]:
df = pd.read_csv('excel_sheets/Grand_NSICaption_Dataset_ManualONLY_Final_DoNOTOpenInExcel_V6.csv')
df.shape

/var/folders/nc/3cln0fc95bz_wm_sppvmqylh0000gn/T/ipykernel_32618/3379865697.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('excel_sheets/Grand_NSICaption_Dataset_ManualONLY_Final_DoNOTOpenInExcel_V6.csv')


(1219007, 10)

In [3]:
# limit to environmental NSI
df = df.loc[df['Main_Model_Estimate'] == '2']
df = df.loc[~(df['Possible_NonEnglish'] > 0.2)]
df.shape

(262649, 10)

In [ ]:
if False:
    sk = Skeletonizer("en_core_web_trf")
    df = explode_nsi(df, sk)
    df.to_pickle("nsi_spans_trf.pkl")        # save right after explode_nsi
else:
    df = pd.read_pickle("nsi_spans_trf.pkl") # load back later                      # one row per NSI span, with skeleton_tags + nsi

In [38]:
import importlib, diverse_sample
importlib.reload(diverse_sample)
from diverse_sample import select_diverse_captions   # re-bind the name(s) you call

In [31]:
before = len(df)
df = df[~df["caption"].str.contains(r"[♩♪♫♬♭♮♯]", regex=True, na=False)].reset_index(drop=True)
df = df[~df["nsi"].str.contains(r"\b(?:music|song|melody|singing|humming|jingle)\b", case=False, regex=True, na=False)].reset_index(drop=True)
print(f"dropped {before - len(df)} music-flagged rows, {len(df)} left")

dropped 1067 music-flagged rows, 264572 left


In [ ]:
selected = select_diverse_captions(df, k=600, weights=(1.0, 0.5, 0.5))
selected.to_excel('excel_sheets/diverses_caption.xlsx', index=False)

In [ ]:
selected = select_diverse_captions(df, k=600, weights=(1.0, 0.2, 0.2))
selected.to_excel('excel_sheets/diverse_captions_lower_weights.xlsx', index=False)

In [ ]:
selected = select_diverse_captions(df, k=600, weights=(1.0, 0.5, 0.5), freq_strength=1.5)
selected.to_excel('excel_sheets/diverse_captions_with_frequency.xlsx', index=False)